In [3]:
# -*- coding: utf-8 -*-
"""
Script 01: Criação da grade de pixels baseada nos dados CHIRPS.
Extrai a extensão correspondente ao Brasil e salva os índices e coordenadas.
"""

import logging
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

# Configurações de Paths utilizando pathlib
CHIRPS_DIR = Path(r"Z:\Analytics & BI\Bases de dados\CHIRPS - chuva")
OUTPUT_DIR = Path(r"Z:\Analytics & BI\Du Cat - Parametric Quoter\Output")
GRADE_DIR = OUTPUT_DIR / "Grade"

# Limites aproximados do Brasil
LAT_MIN, LAT_MAX = -34.5, 6.0
LON_MIN, LON_MAX = -74.5, -34.0

def setup_logging() -> None:
    """Configura o logger para rastreabilidade."""
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s'
    )

def criar_diretorios() -> None:
    """Cria a estrutura de diretórios do Output caso não existam."""
    GRADE_DIR.mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "Matriz").mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "Anos").mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "Final").mkdir(parents=True, exist_ok=True)

def extrair_grade_brasil(nc_path: Path) -> None:
    """
    Lê o NetCDF, filtra para a bounding box do Brasil e cria a grade de pixels.
    
    Args:
        nc_path (Path): Caminho para um arquivo NetCDF de referência.
    """
    logging.info(f"Lendo NetCDF de referência: {nc_path.name}")
    
    try:
        ds = xr.open_dataset(nc_path)
    except Exception as e:
        logging.error(f"Erro ao abrir arquivo NetCDF: {e}")
        raise

    # Filtrar coordenadas para o bounding box brasileiro
    lats = ds.latitude.values
    lons = ds.longitude.values
    
    lat_br = lats[(lats >= LAT_MIN) & (lats <= LAT_MAX)]
    lon_br = lons[(lons >= LON_MIN) & (lons <= LON_MAX)]
    
    logging.info(f"Grade do Brasil definida: {len(lat_br)} latitudes x {len(lon_br)} longitudes.")

    # Criar meshgrid (Grade 2D)
    # np.meshgrid com indexação 'xy' por padrão, lat in rows, lon in cols
    lon_grid, lat_grid = np.meshgrid(lon_br, lat_br)
    
    # Achatar arrays. A ordem de achatamento .ravel() será (Lat, Lon)
    # Fundamental que no Script 03 o achatamento siga a mesma ordem
    pixels_df = pd.DataFrame({
        'LAT': lat_grid.ravel(),
        'LON': lon_grid.ravel()
    })
    
    pixels_df['PIXEL_ID'] = np.arange(len(pixels_df))
    
    # Salvando os arquivos
    grade_file = GRADE_DIR / "grade_pixels.parquet"
    pixels_df.to_parquet(grade_file, index=False)
    
    np.save(GRADE_DIR / "pixel_index.npy", pixels_df['PIXEL_ID'].values)
    np.save(GRADE_DIR / "lat_br.npy", lat_br)
    np.save(GRADE_DIR / "lon_br.npy", lon_br)
    
    logging.info(f"Grade salva com sucesso em {grade_file} com {len(pixels_df)} pixels.")

def main() -> None:
    setup_logging()
    logging.info("Iniciando Etapa 01 - Criar Grade de Pixels")
    criar_diretorios()
    
    # Encontrar um arquivo CHIRPS base (Ex: > 2005)
    chirps_files = list(CHIRPS_DIR.glob("chirps-v2.0.*.days_p05.nc"))
    if not chirps_files:
        logging.error(f"Nenhum arquivo CHIRPS encontrado em {CHIRPS_DIR}")
        return
    
    reference_file = sorted(chirps_files)[0]
    extrair_grade_brasil(reference_file)
    logging.info("Etapa 01 concluída.")

if __name__ == "__main__":
    main()

2026-07-17 17:50:14,580 - INFO - Iniciando Etapa 01 - Criar Grade de Pixels


2026-07-17 17:50:15,472 - INFO - Lendo NetCDF de referência: chirps-v2.0.1990.days_p05.nc
2026-07-17 17:50:17,659 - INFO - Grade do Brasil definida: 810 latitudes x 810 longitudes.
2026-07-17 17:50:32,039 - INFO - Grade salva com sucesso em Z:\Analytics & BI\Du Cat - Parametric Quoter\Output\Grade\grade_pixels.parquet com 656100 pixels.
2026-07-17 17:50:32,066 - INFO - Etapa 01 concluída.


In [4]:
# -*- coding: utf-8 -*-
"""
Script 02: Cruzamento espacial real entre a grade de pixels (polígonos) 
e a malha municipal do IBGE para a geração de pesos normalizados via Matriz Esparsa (CSR).
"""

import logging
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
import scipy.sparse as sparse
from tqdm import tqdm

IBGE_PATH = Path(r"Z:\Analytics & BI\Bases de dados\Shapefile Municipios BR\BR_Municipios_2024.shp")
OUTPUT_DIR = Path(r"Z:\Analytics & BI\Du Cat - Parametric Quoter\Output")
GRADE_DIR = OUTPUT_DIR / "Grade"
MATRIZ_DIR = OUTPUT_DIR / "Matriz"

# Constante do tamanho do pixel em graus (Resolução CHIRPS)
PIXEL_RES = 0.05
# CRS Métrico para cálculo exato de áreas (EPSG:5880 - SIRGAS 2000 / Polyconic para o Brasil)
METRIC_CRS = "EPSG:5880"

def setup_logging() -> None:
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s'
    )

def carregar_dados() -> tuple[gpd.GeoDataFrame, gpd.GeoDataFrame]:
    """Lê shapefile de municípios e cria o GeoDataFrame de pixels."""
    logging.info("Lendo malha municipal IBGE...")
    try:
        mun_gdf = gpd.read_file(IBGE_PATH)
    except Exception as e:
        logging.error(f"Erro ao carregar Shapefile IBGE: {e}")
        raise

    # Garantir nomes de colunas padronizados
    mun_gdf = mun_gdf.rename(columns={'SIGLA_UF': 'UF'})
    
    logging.info("Carregando grade de pixels...")
    grade_df = pd.read_parquet(GRADE_DIR / "grade_pixels.parquet")
    
    logging.info("Criando geometrias de polígonos para os pixels...")
    half_res = PIXEL_RES / 2.0
    # Vetorizando a criação dos bounds
    geoms = [box(lon - half_res, lat - half_res, lon + half_res, lat + half_res)
             for lon, lat in zip(grade_df['LON'], grade_df['LAT'])]
    
    pixels_gdf = gpd.GeoDataFrame(grade_df, geometry=geoms, crs="EPSG:4326")
    
    logging.info("Projetando dados para CRS métrico...")
    mun_gdf = mun_gdf.to_crs(METRIC_CRS)
    pixels_gdf = pixels_gdf.to_crs(METRIC_CRS)
    
    return mun_gdf, pixels_gdf

def construir_matriz_pesos(mun_gdf: gpd.GeoDataFrame, pixels_gdf: gpd.GeoDataFrame) -> None:
    """Cruza pixels e municípios, calcula pesos e exporta a matriz CSR."""
    logging.info("Construindo índice espacial (STRtree)...")
    pixel_sindex = pixels_gdf.sindex
    
    rows = []
    cols = []
    data = []
    
    logging.info("Calculando interseções e pesos...")
    # Iteração sobre os municípios
    for i, mun_row in tqdm(mun_gdf.iterrows(), total=len(mun_gdf), desc="Cruzando Mun x Pixels"):
        mun_geom = mun_row.geometry
        
        # 1. Filtro Bounding Box (O(n log n))
        possible_matches_idx = list(pixel_sindex.intersection(mun_geom.bounds))
        possible_matches = pixels_gdf.iloc[possible_matches_idx]
        
        # 2. Interseção Exata
        exact_matches = possible_matches[possible_matches.intersects(mun_geom)]
        if exact_matches.empty:
            continue
            
        # 3. Cálculo de área em CRS métrico
        intersection_geoms = exact_matches.geometry.intersection(mun_geom)
        inter_areas = intersection_geoms.area
        pixel_areas = exact_matches.geometry.area
        
        # Fração de cobertura
        raw_weights = inter_areas / pixel_areas
        
        # Normalização municipal (Σ pesos = 1)
        normalized_weights = raw_weights / raw_weights.sum()
        
        rows.extend([i] * len(normalized_weights))
        cols.extend(exact_matches['PIXEL_ID'].tolist())
        data.extend(normalized_weights.tolist())

    logging.info("Instanciando Matriz CSR...")
    n_mun = len(mun_gdf)
    n_pixels = len(pixels_gdf)
    csr_mat = sparse.csr_matrix((data, (rows, cols)), shape=(n_mun, n_pixels), dtype=np.float32)
    
    # Salvar matriz CSR
    matriz_file = MATRIZ_DIR / "municipio_pixel.npz"
    sparse.save_npz(matriz_file, csr_mat)
    
    # Salvar index de municípios associados
    df_mun_output = mun_gdf[['CD_MUN', 'NM_MUN', 'UF']].copy()
    df_mun_output['indice'] = df_mun_output.index
    df_mun_output = df_mun_output[['indice', 'CD_MUN', 'NM_MUN', 'UF']]
    
    mun_file = MATRIZ_DIR / "municipios.parquet"
    df_mun_output.to_parquet(mun_file, index=False)
    
    logging.info(f"Matriz esparsa salva em {matriz_file}")
    logging.info(f"Dicionário municipal salvo em {mun_file}")

def main() -> None:
    setup_logging()
    logging.info("Iniciando Etapa 02 - Matriz de Pesos Espaciais")
    mun_gdf, pixels_gdf = carregar_dados()
    construir_matriz_pesos(mun_gdf, pixels_gdf)
    logging.info("Etapa 02 concluída.")

if __name__ == "__main__":
    main()

2026-07-17 17:50:55,313 - INFO - Iniciando Etapa 02 - Matriz de Pesos Espaciais
2026-07-17 17:50:55,313 - INFO - Lendo malha municipal IBGE...
2026-07-17 17:52:47,163 - INFO - Carregando grade de pixels...
2026-07-17 17:52:50,447 - INFO - Criando geometrias de polígonos para os pixels...
2026-07-17 17:53:02,376 - INFO - Projetando dados para CRS métrico...
2026-07-17 17:53:07,140 - INFO - Construindo índice espacial (STRtree)...
2026-07-17 17:53:07,393 - INFO - Calculando interseções e pesos...
Cruzando Mun x Pixels: 100%|██████████| 5573/5573 [10:04<00:00,  9.22it/s]  
2026-07-17 18:03:12,202 - INFO - Instanciando Matriz CSR...
2026-07-17 18:03:18,657 - INFO - Matriz esparsa salva em Z:\Analytics & BI\Du Cat - Parametric Quoter\Output\Matriz\municipio_pixel.npz
2026-07-17 18:03:18,657 - INFO - Dicionário municipal salvo em Z:\Analytics & BI\Du Cat - Parametric Quoter\Output\Matriz\municipios.parquet
2026-07-17 18:03:18,678 - INFO - Etapa 02 concluída.


In [20]:
# -*- coding: utf-8 -*-
"""
Script 03: Processamento diário CHIRPS.
Execução sequencial otimizada para evitar falhas de comunicação entre processos (IPC).
"""

import logging
import gc
from pathlib import Path
import numpy as np
import xarray as xr
import scipy.sparse as sparse
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

# Configurações de Paths
CHIRPS_DIR = Path(r"Z:\Analytics & BI\Bases de dados\CHIRPS - chuva")
OUTPUT_DIR = Path(r"Z:\Analytics & BI\Du Cat - Parametric Quoter\Output")
GRADE_DIR = OUTPUT_DIR / "Grade"
MATRIZ_DIR = OUTPUT_DIR / "Matriz"
ANOS_DIR = OUTPUT_DIR / "Anos"

def setup_logging() -> None:
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def processar_ano(nc_file: Path, matriz: sparse.csr_matrix, df_mun: pd.DataFrame, 
                  lat_br: np.ndarray, lon_br: np.ndarray) -> None:
    """Processa um ano de forma sequencial e segura."""
    ano_str = nc_file.name.split('.')[2] 
    out_parquet = ANOS_DIR / f"municipios_{ano_str}.parquet"
    
    if out_parquet.exists():
        return

    # Abertura com engine 'netcdf4' explícita
    with xr.open_dataset(nc_file, engine='netcdf4') as ds:
        ds_br = ds.sel(latitude=lat_br, longitude=lon_br)
        precip = np.nan_to_num(np.where(ds_br['precip'].values == -9999., np.nan, ds_br['precip'].values), nan=0.0)
        dias = ds_br['time'].values
    
    # Cálculo
    precip_mun_dia = matriz.dot(precip.reshape(precip.shape[0], -1).T)
    
    # Montagem da tabela Arrow (sem pandas)
    n_muns = df_mun.shape[0]
    num_dias = precip.shape[0]
    
    table = pa.Table.from_arrays(
        [
            pa.array(np.repeat(df_mun['CD_MUN'].values, num_dias)),
            pa.array(np.repeat(df_mun['NM_MUN'].values, num_dias)),
            pa.array(np.repeat(df_mun['UF'].values, num_dias)),
            pa.array(np.tile(dias, n_muns)).cast(pa.date32()),
            pa.array(precip_mun_dia.ravel(order='C'), type=pa.float32())
        ],
        names=['CD_MUN', 'NM_MUN', 'UF', 'DATA', 'PRECIP']
    )
    
    pq.write_table(table, out_parquet)
    del precip, precip_mun_dia, table
    gc.collect()

def main() -> None:
    setup_logging()
    logging.info("Carregando matrizes e índices...")
    
    matriz = sparse.load_npz(MATRIZ_DIR / "municipio_pixel.npz")
    df_mun = pd.read_parquet(MATRIZ_DIR / "municipios.parquet")
    lat_br = np.load(GRADE_DIR / "lat_br.npy")
    lon_br = np.load(GRADE_DIR / "lon_br.npy")
    
    arquivos_alvo = [f for f in CHIRPS_DIR.glob("chirps-v2.0.*.days_p05.nc") 
                     if f.name.split('.')[2].isdigit() and int(f.name.split('.')[2]) >= 2006]
    
    logging.info(f"Processando {len(arquivos_alvo)} anos sequencialmente.")
    
    for nc in tqdm(arquivos_alvo, desc="Processando Anos"):
        try:
            processar_ano(nc, matriz, df_mun, lat_br, lon_br)
        except Exception as e:
            logging.error(f"Erro no arquivo {nc.name}: {e}")
            
    logging.info("Etapa 03 concluída com sucesso.")

if __name__ == "__main__":
    import pandas as pd # Import necessário aqui
    main()

2026-07-17 18:19:56,648 - INFO - Carregando matrizes e índices...
2026-07-17 18:19:59,210 - INFO - Processando 21 anos sequencialmente.
Processando Anos: 100%|██████████| 21/21 [45:36<00:00, 130.31s/it] 
2026-07-17 19:05:35,777 - INFO - Etapa 03 concluída com sucesso.


In [21]:
# -*- coding: utf-8 -*-
"""
Script 04: Consolidação dos parquets diários.
Utiliza pyarrow nativo para juntar 40+ milhões de registros sem sobrecarga de memória Pandas.
"""

import logging
from pathlib import Path
import pyarrow.dataset as ds
import pyarrow.parquet as pq

OUTPUT_DIR = Path(r"Z:\Analytics & BI\Du Cat - Parametric Quoter\Output")
ANOS_DIR = OUTPUT_DIR / "Anos"
FINAL_DIR = OUTPUT_DIR / "Final"

def setup_logging() -> None:
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s'
    )

def consolidar_bases() -> None:
    """Lê todos os arquivos Parquet de uma pasta e exporta para um único sem inflar RAM."""
    arquivo_final = FINAL_DIR / "chuva_municipios.parquet"
    
    logging.info(f"Procurando bases anuais em {ANOS_DIR}...")
    parquets = list(ANOS_DIR.glob("municipios_*.parquet"))
    
    if not parquets:
        logging.error("Nenhum arquivo encontrado para consolidar.")
        return
        
    logging.info(f"{len(parquets)} arquivos localizados. Construindo dataset do Arrow...")
    
    try:
        # A API dataset trata múltiplos arquivos como um disco lógico,
        # consumindo quantidades minímas de memória (zero-copy read).
        dataset = ds.dataset(ANOS_DIR, format="parquet")
        
        logging.info("Executando I/O de consolidação (Isso pode demorar alguns minutos)...")
        # Transfere diretamente do disco (leitura) para o disco (escrita)
        table = dataset.to_table()
        pq.write_table(table, arquivo_final)
        
        logging.info(f"Consolidação completa! Base gravada com {table.num_rows:,.0f} linhas.")
        logging.info(f"Destino: {arquivo_final}")
        
    except Exception as e:
        logging.error(f"Erro ao consolidar arquivos: {e}")
        raise

def main() -> None:
    setup_logging()
    logging.info("Iniciando Etapa 04 - Consolidação de Arquivos")
    consolidar_bases()
    logging.info("Etapa 04 concluída. Pipeline finalizado.")

if __name__ == "__main__":
    main()

2026-07-17 19:24:27,656 - INFO - Iniciando Etapa 04 - Consolidação de Arquivos
2026-07-17 19:24:27,659 - INFO - Procurando bases anuais em Z:\Analytics & BI\Du Cat - Parametric Quoter\Output\Anos...
2026-07-17 19:24:27,981 - INFO - 21 arquivos localizados. Construindo dataset do Arrow...
2026-07-17 19:24:30,838 - INFO - Executando I/O de consolidação (Isso pode demorar alguns minutos)...
2026-07-17 19:41:05,538 - INFO - Consolidação completa! Base gravada com 41,379,525 linhas.
2026-07-17 19:41:05,539 - INFO - Destino: Z:\Analytics & BI\Du Cat - Parametric Quoter\Output\Final\chuva_municipios.parquet
2026-07-17 19:41:05,544 - INFO - Etapa 04 concluída. Pipeline finalizado.


In [2]:
from pathlib import Path
import pandas as pd

# ============================================================
# Caminhos
# ============================================================

arquivo_entrada = Path(
    r"Z:\Analytics & BI\Du Cat - Parametric Quoter\Output\Final\chuva_municipios.parquet"
)

arquivo_saida = Path(
    r"Z:\Analytics & BI\Du Cat - Parametric Quoter\Output\Final\chuva_municipios_sample.parquet"
)

# ============================================================
# Leitura
# ============================================================

df = pd.read_parquet(arquivo_entrada)

# ============================================================
# Procura automaticamente uma amostra <= 20 MB
# ============================================================

limite_mb = 20

fracao = 0.05

while True:

    amostra = df.sample(frac=fracao, random_state=42)

    amostra.to_parquet(
        arquivo_saida,
        index=False,
        compression="snappy"
    )

    tamanho_mb = arquivo_saida.stat().st_size / 1024**2

    print(f"{fracao:.1%} -> {tamanho_mb:.2f} MB")

    if tamanho_mb <= limite_mb:
        break

    fracao *= 0.9

print("\nArquivo criado:")
print(arquivo_saida)
print(f"Tamanho: {tamanho_mb:.2f} MB")
print(f"Linhas: {len(amostra):,}")

5.0% -> 16.52 MB

Arquivo criado:
Z:\Analytics & BI\Du Cat - Parametric Quoter\Output\Final\chuva_municipios_sample.parquet
Tamanho: 16.52 MB
Linhas: 2,068,976
